In [1]:
# [Setup]: 

# Ran the following in Anaconda Prompt:
# git clone https://github.com/burke86/deepdisc.git
# cd deepdisc
# conda create -n deepdisc python=3.10 -y
# conda activate deepdisc
# pip install setuptools==67.8.0
# pip install pybind11
# NOTE: scarlet skipped, does not compile on Windows (optional dependency, not needed for detection)

# Ran the following in Anaconda Prompt:
# nvidia-smi

# If Version CUDA is not 12.1, install older versions to be compatible with torch and relevant packages:
# Ran the following in Anaconda Prompt:
# conda activate deepdisc
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Ran the following in Anaconda Prompt:
# pip install --no-build-isolation git+https://github.com/facebookresearch/detectron2.git
# mkdir taufit
# echo __version__ = "0.1.0" > taufit\version.py
# echo. > requirements.txt
# pip install --no-build-isolation -e .
# pip install --no-deps --no-build-isolation .
# pip install opencv-python
# pip install scikit-image
# pip install ipympl
# pip install ipywidgets
# pip install astropy photutils matplotlib pandas ipykernel astroquery
# python -m ipykernel install --user --name deepdisc --display-name "Python (deepdisc)"

# Now, in VSCodium, click top right for environment -> "Choose another Kernel" -> "deepdisc"

In [ ]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 1

%matplotlib widget

import sys
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits as astrofits
from astropy.wcs import WCS
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# Paths
BASE_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN")

SCRIPTS_DIR = BASE_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from plate_scan_utils import (
    detect_sources, detect_plate_errors, classify_plate_quality,
    annotate_errors, categorize_plate, fair_plate_features,
    get_plate_catalog_gaia, get_plate_catalog_simbad, get_plate_catalog_apass,
    match_detected_sources_gaia, match_detected_sources_simbad, match_detected_sources_apass,
    match_against_paradigm, update_name_cache, resolve_names, suggest_name_at,
    compute_linearity_fit, compute_fwhm_for_sources,
)

cutout_dir = BASE_DIR / "data" / "V_CrA" / "cutouts"
manifest_path = cutout_dir.parent / "plate_manifest.csv"

# Caches (scan results, paradigm marks, Gaia/SIMBAD/APASS caches) stay
# under data/V_CrA/02_B -- this is internal bookkeeping, not a deliverable.
scan_cache_dir = BASE_DIR / "data" / "V_CrA" / "02_B" / "scan_cache"
scan_cache_dir.mkdir(parents=True, exist_ok=True)
PLATE_DB_FILE = scan_cache_dir / "plate_db_pytorch.pkl"
PARADIGM_FILE = scan_cache_dir / "paradigm_marks.pkl"
GAIA_CACHE_FILE = scan_cache_dir / "gaia_cache.pkl"
SIMBAD_CACHE_FILE = scan_cache_dir / "simbad_cache.pkl"
APASS_CACHE_FILE = scan_cache_dir / "apass_cache.pkl"
NAME_CACHE_FILE = scan_cache_dir / "gaia_name_cache.pkl"

output_dir = BASE_DIR / "outputs" / "V_CrA"
img_dir    = output_dir / "images"
mask_dir   = output_dir / "masks"
labels_dir = output_dir / "labels"
img_dir.mkdir(parents=True, exist_ok=True)
mask_dir.mkdir(parents=True, exist_ok=True)
labels_dir.mkdir(parents=True, exist_ok=True)

cutouts = sorted(list(cutout_dir.glob("*.fits")) + list(cutout_dir.glob("*.fit")))
print(f"Found {len(cutouts)} cutouts to process")
if len(cutouts) == 0:
    raise RuntimeError("No FITS files found in cutout directory")

PRESCAN_WORKERS = 8
QUALITY_VERSION = 1  # bump to force every cached plate to be re-derived

# Plate limiting-magnitude lookup, same source as 02_A
plate_limit_lookup = {}
if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)
    if "filename" in manifest_df.columns:
        for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Reading plate manifest", unit="row"):
            fname = row.get("filename")
            if not (isinstance(fname, str) and fname):
                continue
            a, t = row.get("lim_mag_apass"), row.get("lim_mag_atlas")
            plate_limit_lookup[Path(fname).name] = {
                "lim_mag_apass": float(a) if pd.notna(a) else None,
                "lim_mag_atlas": float(t) if pd.notna(t) else None,
            }
print(f"Loaded plate limits for {len(plate_limit_lookup)} manifest rows.")

def get_plate_limits(fits_path):
    entry = plate_limit_lookup.get(fits_path.name)
    return (entry["lim_mag_apass"], entry["lim_mag_atlas"]) if entry else (None, None)

_apass_vals = [v["lim_mag_apass"] for v in plate_limit_lookup.values() if v.get("lim_mag_apass") is not None]
_atlas_vals = [v["lim_mag_atlas"] for v in plate_limit_lookup.values() if v.get("lim_mag_atlas") is not None]
LIM_MAG_MEDIAN_APASS = float(np.median(_apass_vals)) if _apass_vals else None
LIM_MAG_MEDIAN_ATLAS = float(np.median(_atlas_vals)) if _atlas_vals else None

# Plate database (source detections + quality only -- no naming here yet)
if PLATE_DB_FILE.exists():
    with open(PLATE_DB_FILE, "rb") as fp:
        plate_db = pickle.load(fp)
else:
    plate_db = {}

# Gaia / SIMBAD / APASS / name caches
if GAIA_CACHE_FILE.exists():
    with open(GAIA_CACHE_FILE, "rb") as f:
        gaia_cache = pickle.load(f)
else:
    gaia_cache = {}

if SIMBAD_CACHE_FILE.exists():
    with open(SIMBAD_CACHE_FILE, "rb") as f:
        simbad_cache = pickle.load(f)
else:
    simbad_cache = {}

if APASS_CACHE_FILE.exists():
    with open(APASS_CACHE_FILE, "rb") as f:
        apass_cache = pickle.load(f)
else:
    apass_cache = {}

if NAME_CACHE_FILE.exists():
    with open(NAME_CACHE_FILE, "rb") as f:
        gaia_name_cache = pickle.load(f)
else:
    gaia_name_cache = {}

def save_gaia_cache():
    with open(GAIA_CACHE_FILE, "wb") as f:
        pickle.dump(gaia_cache, f)

def save_simbad_cache():
    with open(SIMBAD_CACHE_FILE, "wb") as f:
        pickle.dump(simbad_cache, f)

def save_apass_cache():
    with open(APASS_CACHE_FILE, "wb") as f:
        pickle.dump(apass_cache, f)

def save_name_cache():
    with open(NAME_CACHE_FILE, "wb") as f:
        pickle.dump(gaia_name_cache, f)

def _scan_one_plate(f):
    cached = plate_db.get(str(f))
    if cached is not None and cached.get("quality_version") == QUALITY_VERSION:
        return (f, "skip", None)
    try:
        sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(f)
        n = 0 if sources is None else len(sources)
        errors = detect_plate_errors(data if sources is not None else astrofits.getdata(f).astype(float))
        lim_apass, lim_atlas = get_plate_limits(f)
        quality = classify_plate_quality(
            errors, n, None, lim_apass, lim_atlas, LIM_MAG_MEDIAN_APASS, LIM_MAG_MEDIAN_ATLAS
        )
        entry = {
            "n": n, "quality": quality, "quality_version": QUALITY_VERSION,
            "errors": errors, "sources": sources, "x_col": x_col, "y_col": y_col,
            "human_verdict": cached.get("human_verdict") if cached else None,
        }
        return (f, "new", entry)
    except Exception as e:
        entry = {
            "n": 0, "quality": "defective", "quality_version": QUALITY_VERSION,
            "errors": {}, "sources": None, "x_col": None, "y_col": None,
            "human_verdict": None, "error": str(e),
        }
        return (f, "new", entry)

def prescan(plates=None, max_workers=PRESCAN_WORKERS):
    target_plates = plates if plates is not None else cutouts
    total = max(len(target_plates), 1)
    errors_seen = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_scan_one_plate, f): f for f in target_plates}
        pbar = tqdm(as_completed(futures), total=total, desc="Prescanning plates", unit="plate")
        for fut in pbar:
            f = futures[fut]
            try:
                f_ret, action, payload = fut.result()
            except Exception as e:
                errors_seen.append(f"{f.name}: {e}")
                continue
            if action == "new":
                plate_db[str(f)] = payload
                if "error" in payload:
                    errors_seen.append(f"{f.name}: {payload['error']}")
            pbar.set_postfix(cached=len(plate_db))

    with open(PLATE_DB_FILE, "wb") as fp:
        pickle.dump(plate_db, fp)

    print(f"Prescan complete: {len(plate_db)} plates cached.")
    if errors_seen:
        print(f"{len(errors_seen)} plate(s) had errors during scanning:")
        for msg in errors_seen[:20]:
            print(f"  [SCAN ERROR] {msg}")
        if len(errors_seen) > 20:
            print(f"  ...and {len(errors_seen) - 20} more.")

prescan()

Found 5398 cutouts to process


Reading plate manifest:   0%|          | 0/7218 [00:00<?, ?row/s]

Loaded plate limits for 5398 manifest rows.


Prescanning plates:   0%|          | 0/5398 [00:00<?, ?plate/s]

Prescan complete: 5398 plates cached.


In [7]:
# NOTE : 02_B_PyTorch_Algorithm.ipynb - Cell 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
from tqdm.notebook import tqdm
import random
import json
import copy

BOX_HALF = 8
VAL_SPLIT = 0.15
MAX_EPOCHS = 60 # Upper bound, early stopping will likely stop before this
EARLY_STOP_PATIENCE = 8 # Stop if val loss hasn't improved in this many epochs

def crop_to_match(x, ref):
    _, _, h, w = ref.shape
    return x[:, :, :h, :w]

def pad_to_multiple(img, multiple=16):
    _, h, w = img.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    return F.pad(img, (0, pad_w, 0, pad_h))

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU())
        self.enc1, self.enc2, self.enc3 = block(1, 32), block(32, 64), block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(128, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, stride=2), block(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, stride=2), block(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, stride=2), block(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(m), crop_to_match(e3, self.up3(m))], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), crop_to_match(e2, self.up2(d3))], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), crop_to_match(e1, self.up1(d2))], dim=1))
        return torch.sigmoid(self.out(d1))

class StarSegDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths, self.mask_paths = image_paths, mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        img = pad_to_multiple(torch.tensor(img).unsqueeze(0))
        mask = pad_to_multiple(torch.tensor(mask).unsqueeze(0))
        return img, mask

def _points_to_mask(shape, xs, ys, box_half=BOX_HALF):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    for x, y in zip(xs, ys):
        x0, x1 = max(0, int(x - box_half)), min(w, int(x + box_half))
        y0, y1 = max(0, int(y - box_half)), min(h, int(y + box_half))
        mask[y0:y1, x0:x1] = 255
    return mask

def _save_image_mask_pair(fits_path, xs, ys, out_img_dir, out_mask_dir):
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)
    mask = _points_to_mask(img_8bit.shape, xs, ys)

    img_path = out_img_dir / (fits_path.stem + ".png")
    mask_path = out_mask_dir / (fits_path.stem + "_mask.png")
    cv2.imwrite(str(img_path), img_8bit)
    cv2.imwrite(str(mask_path), mask)
    return img_path, mask_path

def _identify_sources(fits_path, xs, ys, known_labels=None):
    """Guesses a name for each (x, y) source on this plate via WCS +
    Gaia/SIMBAD/paradigm crossmatch -- the same approach 02_A uses. This is
    a catalog lookup, NOT something the UNet learns; the segmentation
    network only ever predicts star/not-star. known_labels, if given, is a
    parallel list of user-provided names (from paradigm marking) that take
    priority over the automatic guess wherever non-empty."""
    hdr = astrofits.getheader(fits_path)
    wcs = WCS(hdr)
    xs_arr, ys_arr = np.array(xs, dtype=float), np.array(ys, dtype=float)
    ra, dec = wcs.pixel_to_world_values(xs_arr, ys_arr)
    ra, dec = np.array(ra, dtype=float), np.array(dec, dtype=float)

    data = astrofits.getdata(fits_path)
    try:
        gaia_catalog = get_plate_catalog_gaia(wcs, data.shape, gaia_cache)
        simbad_catalog = get_plate_catalog_simbad(wcs, data.shape, simbad_cache)
        gaia_names = match_detected_sources_gaia(ra, dec, gaia_catalog)
        simbad_names = match_detected_sources_simbad(ra, dec, simbad_catalog)
        update_name_cache(gaia_names, simbad_names, gaia_name_cache)
        resolved = resolve_names(gaia_names, simbad_names, gaia_name_cache)
    except Exception as e:
        print(f"[IDENTIFY ERROR] {fits_path.name}: {e}")
        resolved = ["Unknown"] * len(xs_arr)

    paradigm_names = match_against_paradigm(ra, dec, paradigm_marks)
    final_names = [p if p != "Unknown" else r for p, r in zip(paradigm_names, resolved)]

    if known_labels is not None:
        final_names = [k if k else f for k, f in zip(known_labels, final_names)]

    return [
        {"x": float(x), "y": float(y), "ra": float(r), "dec": float(d), "name": n}
        for x, y, r, d, n in zip(xs_arr, ys_arr, ra, dec, final_names)
    ]

def _save_labels(fits_path, records, out_labels_dir):
    label_path = out_labels_dir / (fits_path.stem + "_labels.json")
    with open(label_path, "w") as f:
        json.dump(records, f, indent=2)
    return label_path

def build_dataset_and_train():
    image_paths, mask_paths = [], []
    n_named_total, n_source_total = 0, 0

    # 1. Ground truth: your human-corrected paradigm plates. Labels the
    #    user typed in take priority; anything left "Unknown" gets one
    #    more automatic crossmatch attempt.
    paradigm_items = [(f_str, marks) for f_str, marks in paradigm_marks.items() if marks]
    for f_str, marks in tqdm(paradigm_items, desc="Building paradigm (ground-truth) plates", unit="plate"):
        fp = Path(f_str)
        xs, ys = [m["x"] for m in marks], [m["y"] for m in marks]
        known = [m.get("label", "") for m in marks]
        ip, mp = _save_image_mask_pair(fp, xs, ys, img_dir, mask_dir)
        image_paths.append(ip); mask_paths.append(mp)

        records = _identify_sources(fp, xs, ys, known_labels=known)
        _save_labels(fp, records, labels_dir)
        n_named_total += sum(1 for r in records if r["name"] != "Unknown")
        n_source_total += len(records)
    print(f"{len(image_paths)} paradigm (ground-truth) plate(s) added.")

    # 2. Pseudo-labels: every other cached plate that passed the quality
    #    bar (algorithm-detected sources used as weak labels), skipping
    #    anything defective/too_many_errors or explicitly rejected, and
    #    skipping plates already added above. Identity is guessed the
    #    same way as for paradigm plates.
    pseudo_items = [
        (f_str, meta) for f_str, meta in plate_db.items()
        if not (f_str in paradigm_marks and paradigm_marks[f_str])
        and meta.get("human_verdict") != "rejected"
        and meta.get("quality") in ("good_match", "fair")
        and meta.get("sources") is not None
    ]
    n_pseudo = 0
    for f_str, meta in tqdm(pseudo_items, desc="Building pseudo-labeled plates", unit="plate"):
        fp = Path(f_str)
        xs = list(meta["sources"][meta["x_col"]])
        ys = list(meta["sources"][meta["y_col"]])
        ip, mp = _save_image_mask_pair(fp, xs, ys, img_dir, mask_dir)
        image_paths.append(ip); mask_paths.append(mp)
        n_pseudo += 1

        records = _identify_sources(fp, xs, ys)
        _save_labels(fp, records, labels_dir)
        n_named_total += sum(1 for r in records if r["name"] != "Unknown")
        n_source_total += len(records)
    print(f"{n_pseudo} pseudo-labeled plate(s) added (quality-filtered algorithm detections).")
    print(f"Identity crossmatch: {n_named_total} of {n_source_total} source(s) across all plates got a name "
          f"(rest saved as 'Unknown'). Per-plate results in {labels_dir}")

    save_gaia_cache(); save_simbad_cache(); save_name_cache()

    if len(image_paths) < 4:
        print("Too few plates to train on -- label more stars on your paradigm plate(s) or approve more 'fair' plates.")
        return None

    combined = list(zip(image_paths, mask_paths))
    random.shuffle(combined)
    n_val = max(1, int(len(combined) * VAL_SPLIT))
    val_pairs, train_pairs = combined[:n_val], combined[n_val:]

    train_ds = StarSegDataset([p[0] for p in train_pairs], [p[1] for p in train_pairs])
    val_ds = StarSegDataset([p[0] for p in val_pairs], [p[1] for p in val_pairs])
    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCELoss()

    train_loss_history = []
    val_loss_history = []
    best_val_loss = float("inf")
    best_state_dict = None
    epochs_without_improvement = 0
    stopped_early_at = None

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_loss_sum = 0
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, masks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_loss_sum += loss.item()
        train_loss_avg = train_loss_sum / max(len(train_loader), 1)

        model.eval()
        val_loss_sum = 0
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]"):
                imgs, masks = imgs.to(device), masks.to(device)
                val_loss_sum += loss_fn(model(imgs), masks).item()
        val_loss_avg = val_loss_sum / max(len(val_loader), 1)

        train_loss_history.append(train_loss_avg)
        val_loss_history.append(val_loss_avg)
        print(f"Epoch {epoch+1}: Train Loss={train_loss_avg:.4f} | Val Loss={val_loss_avg:.4f}")

        if val_loss_avg < best_val_loss - 1e-5:
            best_val_loss = val_loss_avg
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                stopped_early_at = epoch + 1
                print(f"No val-loss improvement for {EARLY_STOP_PATIENCE} epochs -- stopping early at epoch {stopped_early_at}.")
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    model_path = output_dir / "pytorch_trial_1.pth"
    torch.save(model.state_dict(), model_path)
    if stopped_early_at is not None:
        print(f"Training stopped early at epoch {stopped_early_at} (best val loss {best_val_loss:.4f}). Model saved to {model_path}")
    else:
        print(f"Training ran the full {MAX_EPOCHS} epochs (best val loss {best_val_loss:.4f}). Model saved to {model_path}")

    # Loss curve : both lines are now per-batch AVERAGES, so they're on
    # the same scale and directly comparable, unlike the earlier summed
    # version.
    epochs_range = list(range(1, len(train_loss_history) + 1))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs_range, train_loss_history, marker="o", label="Train Loss")
    ax.plot(epochs_range, val_loss_history, marker="o", label="Val Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BCE Loss (average per batch)")
    ax.set_title("Training Progress")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return {"train_loss": train_loss_history, "val_loss": val_loss_history, "model_path": model_path}

In [ ]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 3

import ipywidgets as widgets
from IPython.display import display, clear_output
from threading import Thread
import json

REVIEW_MAX_GROUPS = 20
PARADIGM_CLICK_PX = 15

if PARADIGM_FILE.exists():
    with open(PARADIGM_FILE, "rb") as f:
        _loaded = pickle.load(f)
else:
    _loaded = {}

paradigm_marks = {}
for f_str, entries in _loaded.items():
    fixed = []
    for e in entries:
        if isinstance(e, dict):
            fixed.append(e)
        else:
            x, y = e
            fixed.append({"x": float(x), "y": float(y), "ra": None, "dec": None, "label": "", "source": "legacy"})
    paradigm_marks[f_str] = fixed

def save_paradigm_marks():
    with open(PARADIGM_FILE, "wb") as f:
        pickle.dump(paradigm_marks, f)

def _load_algorithm_labels(fits_path):
    label_path = labels_dir / (fits_path.stem + "_labels.json")
    if not label_path.exists():
        return None
    with open(label_path, "r") as f:
        return json.load(f)

# Review queue

_fair_cluster_of = {}
_fair_cluster_members = {}

def _fair_plate_signature(meta):
    errs = meta.get("errors", {})
    total_defects = (
        len(errs.get("scratches", [])) + len(errs.get("trailing", []))
        + len(errs.get("saturation", [])) + len(errs.get("dust", []))
        + (1 if errs.get("edge") else 0)
    )
    dead_bucket = min(int(errs.get("dead_zone_fraction", 0.0) * 20), 3)
    sat_bucket = min(int(errs.get("saturation_area_fraction", 0.0) * 20), 3)
    n_bucket = min(int(meta.get("n", 0)) // 10, 10)
    return (min(total_defects, 8), dead_bucket, sat_bucket, n_bucket)

def _cluster_fair_plates(max_groups=REVIEW_MAX_GROUPS):
    global _fair_cluster_of, _fair_cluster_members
    _fair_cluster_of, _fair_cluster_members = {}, {}

    fair_items = [
        (f_str, meta) for f_str, meta in plate_db.items()
        if meta.get("quality") == "fair" and meta.get("human_verdict") is None
        and meta.get("sources") is not None
    ]
    if not fair_items:
        return []

    groups = {}
    for f_str, meta in fair_items:
        sig = _fair_plate_signature(meta)
        groups.setdefault(sig, []).append(f_str)

    ranked = sorted(groups.items(), key=lambda kv: -len(kv[1]))[:max_groups]

    representatives = []
    for cluster_id, (sig, member_files) in enumerate(ranked):
        for mf in member_files:
            _fair_cluster_of[mf] = cluster_id
        _fair_cluster_members[cluster_id] = member_files
        member_files_sorted = sorted(member_files, key=lambda fs: plate_db[fs].get("n", 0))
        rep = member_files_sorted[len(member_files_sorted) // 2]
        representatives.append(rep)

    return representatives

review_plate_dropdown = widgets.Dropdown(description="Needs review:")
review_status = widgets.Label(value="")
approve_btn = widgets.Button(description="Approve (treat as good)", button_style="success")
reject_btn = widgets.Button(description="Reject (exclude from shortlist)", button_style="danger")
review_instructions = widgets.HTML(
    "<small>Up to 20 representative 'fair'-quality plates -- one per group of plates "
    "sharing the same defect/size signature, largest groups first so each decision "
    "clears as many plates as possible. <b>Approve</b> treats this representative AND "
    "its whole group as good; <b>Reject</b> excludes them from the paradigm-plate "
    "shortlist below.</small>"
)

def _pending_review_plates():
    representatives = _cluster_fair_plates()
    items = []
    for f_str in representatives:
        meta = plate_db.get(f_str)
        if meta is None:
            continue
        f = Path(f_str)
        errs = meta.get("errors", {})
        cluster_id = _fair_cluster_of.get(f_str)
        n_members = len(_fair_cluster_members.get(cluster_id, [f_str]))
        label = (
            f"{f.name} | represents {n_members} plate(s) | {meta['n']} sources | "
            f"defects: scr{len(errs.get('scratches', []))} "
            f"trail{len(errs.get('trailing', []))} "
            f"sat{len(errs.get('saturation', []))} "
            f"dust{len(errs.get('dust', []))}"
        )
        items.append((n_members, label, f))
    items.sort(key=lambda t: -t[0])
    return [(label, f) for _, label, f in items]

def _refresh_review_queue():
    pending = _pending_review_plates()
    review_plate_dropdown.options = pending
    review_status.value = f"{len(pending)} representative plate(s) awaiting review."

def _approve_plate(_):
    fp = review_plate_dropdown.value
    if fp is None: return
    for m in _fair_cluster_members.get(_fair_cluster_of.get(str(fp)), [str(fp)]):
        if m in plate_db:
            plate_db[m]["human_verdict"] = "approved"
    with open(PLATE_DB_FILE, "wb") as f: pickle.dump(plate_db, f)
    _refresh_review_queue()
    refresh_filter_options()
    rebuild_paradigm_dropdown()

def _reject_plate(_):
    fp = review_plate_dropdown.value
    if fp is None: return
    for m in _fair_cluster_members.get(_fair_cluster_of.get(str(fp)), [str(fp)]):
        if m in plate_db:
            plate_db[m]["human_verdict"] = "rejected"
    with open(PLATE_DB_FILE, "wb") as f: pickle.dump(plate_db, f)
    _refresh_review_queue()
    refresh_filter_options()
    rebuild_paradigm_dropdown()

approve_btn.on_click(_approve_plate)
reject_btn.on_click(_reject_plate)

# Filter/category infrastructure

CATEGORY_DISPLAY = {"ideal": "Ideal Image", "good_no_target": "Good Image", "defective_no_target": "Defective Image"}

category_filter = widgets.Dropdown(description="Filter:")
search_box = widgets.Text(description="Search:", placeholder="filter by filename")
paradigm_plate_dropdown = widgets.Dropdown(description="Paradigm plate:")
paradigm_dropdown_status = widgets.Label(value="")

def filter_plates(cat_value, query):
    query = query.strip().lower()
    matches = []
    for f_str, meta in plate_db.items():
        if meta.get("human_verdict") == "rejected":
            continue
        cat = categorize_plate(meta)
        if cat_value != "all" and cat != cat_value:
            continue
        f = Path(f_str)
        if query and query not in f.name.lower():
            continue
        label = f"{f.name} | {CATEGORY_DISPLAY.get(cat, cat)} | {meta['n']} sources"
        matches.append((label, f))
    return matches

def refresh_filter_options():
    counts = {"ideal": 0, "good_no_target": 0, "defective_no_target": 0}
    total = 0
    for meta in plate_db.values():
        if meta.get("human_verdict") == "rejected":
            continue
        cat = categorize_plate(meta)
        counts[cat] = counts.get(cat, 0) + 1
        total += 1
    new_options = [
        (f"all ({total})", "all"),
        (f"{CATEGORY_DISPLAY['ideal']} ({counts['ideal']})", "ideal"),
        (f"{CATEGORY_DISPLAY['good_no_target']} ({counts['good_no_target']})", "good_no_target"),
        (f"{CATEGORY_DISPLAY['defective_no_target']} ({counts['defective_no_target']})", "defective_no_target"),
    ]
    cur_val = category_filter.value
    category_filter.options = new_options
    category_filter.value = cur_val if cur_val in [o[1] for o in new_options] else "all"

def rebuild_paradigm_dropdown(*args):
    matches = filter_plates(category_filter.value, search_box.value)
    total_matches = len(matches)
    options = matches[:500]
    paradigm_plate_dropdown.options = options
    if total_matches > 500:
        paradigm_dropdown_status.value = f"Showing first 500 of {total_matches} matches -- narrow with Search or the filter above."
    else:
        paradigm_dropdown_status.value = f"{total_matches} matches"

category_filter.observe(rebuild_paradigm_dropdown, names="value")
search_box.observe(rebuild_paradigm_dropdown, names="value")

# ---------- Shared interactive plate viewer ----------

error_toggle = widgets.ToggleButtons(options=["Show Errors", "Hide Errors"], value="Show Errors", description="Errors:")
apass_overlay_toggle = widgets.ToggleButtons(
    options=["Show APASS Reference", "Hide APASS Reference"], value="Show APASS Reference", description="APASS:"
)

shared_plot_output = widgets.Output()
shared_info_output = widgets.Output()

_viewer_state = {
    "fig": None, "ax": None, "cids": [], "fits_path": None, "mode": None,
    "scatter": None, "highlight_marker": None, "annotation": None,
    "entries": [], "wcs": None,
}

def _teardown_shared_plot():
    fig = _viewer_state.get("fig")
    if fig is not None:
        for cid in _viewer_state.get("cids", []):
            try:
                fig.canvas.mpl_disconnect(cid)
            except Exception:
                pass
        plt.close(fig)
    _viewer_state.update({"fig": None, "ax": None, "cids": [], "highlight_marker": None,
                           "annotation": None, "scatter": None})

def _entry_xy():
    entries = _viewer_state["entries"]
    xs = [e["x"] for e in entries]
    ys = [e["y"] for e in entries]
    return np.array(xs), np.array(ys)

def _star_label_text(i):
    e = _viewer_state["entries"][i]
    if _viewer_state["mode"] == "editable":
        lines = [e["label"] if e.get("label") else "(unlabeled)"]
        if e.get("ra") is not None and e.get("dec") is not None:
            lines.append(f"RA={e['ra']:.5f} Dec={e['dec']:.5f}")
        lines.append(f"source={e.get('source', 'manual')}")
    else:
        lines = [e.get("name", "Unknown")]
        if np.isfinite(e.get("inst_mag", np.nan)):
            lines.append(f"inst={e['inst_mag']:.2f}")
    return "\n".join(lines)

def _highlight_star(idx):
    ax = _viewer_state["ax"]
    if ax is None or idx is None:
        return
    e = _viewer_state["entries"][idx]
    hl = _viewer_state["highlight_marker"]
    hl.set_data([e["x"]], [e["y"]])
    hl.set_visible(True)
    ann = _viewer_state["annotation"]
    ann.xy = (e["x"], e["y"])
    ann.set_text(_star_label_text(idx))
    ann.set_visible(True)
    _viewer_state["fig"].canvas.draw_idle()
    with shared_info_output:
        clear_output(wait=True)
        print(_star_label_text(idx).replace("\n", "   |   "))

def _redraw_scatter():
    ax = _viewer_state["ax"]
    if _viewer_state["scatter"] is not None:
        _viewer_state["scatter"].remove()
    xs, ys = _entry_xy()
    if _viewer_state["mode"] == "editable":
        colors = ["limegreen" if e.get("label") else "orange" for e in _viewer_state["entries"]]
    else:
        colors = "red"
    _viewer_state["scatter"] = ax.scatter(xs, ys, s=60, facecolors="none", edgecolors=colors,
                                           linewidths=1.3, picker=True, pickradius=8)
    ax.figure.canvas.draw_idle()

def _on_pick(event):
    if event.artist != _viewer_state["scatter"]:
        return
    _highlight_star(event.ind[0])

def _open_label_editor(entry):
    label_type = widgets.ToggleButtons(options=["Object ID", "GAIA ID"], value="Object ID", description="Type:")
    label_box = widgets.Text(
        value=entry["label"] if entry.get("label") and entry["label"] != "Unknown" else "",
        description="Label:", placeholder="looking up suggestion... (or just start typing)"
    )
    confirm_btn = widgets.Button(description="Confirm", button_style="success")
    skip_btn = widgets.Button(description="Skip", button_style="")

    def _confirm(_):
        raw = label_box.value.strip()
        if label_type.value == "GAIA ID":
            digits = raw.replace("Gaia", "").strip()
            if digits.isdigit():
                final_label = f"Gaia {digits}"
            else:
                with shared_info_output:
                    clear_output(wait=True)
                    print(f"'{raw}' isn't a numeric Gaia source_id -- enter digits only.")
                return
        else:
            final_label = raw
        entry["label"] = final_label
        entry["source"] = "manual"
        save_paradigm_marks()
        _redraw_scatter()
        with shared_info_output:
            clear_output(wait=True)
            print(f"Saved: {final_label}")

    def _skip(_):
        with shared_info_output:
            clear_output(wait=True)

    confirm_btn.on_click(_confirm)
    skip_btn.on_click(_skip)

    with shared_info_output:
        clear_output(wait=True)
        print(f"Pixel ({entry['x']:.1f}, {entry['y']:.1f})  ->  RA={entry['ra']:.5f}, Dec={entry['dec']:.5f}")
        if not entry.get("label") or entry["label"] == "Unknown":
            print("Looking up a name suggestion in the background...")
        display(widgets.HBox([label_type, label_box, confirm_btn, skip_btn]))

    def _do_lookup():
        if entry.get("label") and entry["label"] != "Unknown":
            return
        try:
            suggestion, src = suggest_name_at(entry["ra"], entry["dec"])
        except Exception as e:
            suggestion, src = "Unknown", f"error: {e}"
        if not label_box.value.strip():
            suggested_mode = "GAIA ID" if suggestion.startswith("Gaia ") else "Object ID"
            label_type.value = suggested_mode
            label_box.value = suggestion.replace("Gaia ", "") if suggested_mode == "GAIA ID" else suggestion
        with shared_info_output:
            print(f"Auto-suggested: '{suggestion}' (source: {src})")

    Thread(target=_do_lookup, daemon=True).start()

def _on_paradigm_click(event):
    if event.inaxes != _viewer_state["ax"] or event.xdata is None:
        return
    entries = _viewer_state["entries"]
    wcs = _viewer_state["wcs"]

    if event.button == 3:
        if not entries:
            return
        xs, ys = _entry_xy()
        d = np.hypot(xs - event.xdata, ys - event.ydata)
        j = int(np.argmin(d))
        if d[j] > PARADIGM_CLICK_PX:
            with shared_info_output:
                clear_output(wait=True); print("No marker close enough to remove.")
            return
        entries.pop(j)
        paradigm_marks[str(_viewer_state["fits_path"])] = entries
        save_paradigm_marks()
        _redraw_scatter()
        with shared_info_output:
            clear_output(wait=True); print("Removed marker.")
        return

    if entries:
        xs, ys = _entry_xy()
        d = np.hypot(xs - event.xdata, ys - event.ydata)
        j = int(np.argmin(d))
        if d[j] <= PARADIGM_CLICK_PX:
            _highlight_star(j)
            _open_label_editor(entries[j])
            return

    x, y = float(event.xdata), float(event.ydata)
    ra, dec = wcs.pixel_to_world_values(x, y)
    ra, dec = float(np.array(ra)), float(np.array(dec))
    entry = {"x": x, "y": y, "ra": ra, "dec": dec, "label": "", "source": "manual"}
    entries.append(entry)
    paradigm_marks[str(_viewer_state["fits_path"])] = entries
    save_paradigm_marks()
    _redraw_scatter()
    _open_label_editor(entry)

def _overlay_apass_reference(ax, wcs, data_shape):
    """Non-interactive marker layer showing APASS-catalogued star
    positions on top of whatever DAOStarFinder detected -- purely a
    visual sanity check, doesn't feed into paradigm marking, training,
    or anything else. Cached by field center, so repeat views of the
    same plate don't re-query."""
    try:
        apass_rows = get_plate_catalog_apass(wcs, data_shape, apass_cache)
        save_apass_cache()
    except Exception as e:
        with shared_info_output:
            print(f"[APASS OVERLAY ERROR] {e}")
        return 0

    if not apass_rows:
        return 0

    ra_arr = np.array([r[0] for r in apass_rows])
    dec_arr = np.array([r[1] for r in apass_rows])
    x_arr, y_arr = wcs.world_to_pixel_values(ra_arr, dec_arr)
    x_arr, y_arr = np.array(x_arr, dtype=float), np.array(y_arr, dtype=float)

    h, w = data_shape
    in_bounds = (x_arr >= 0) & (x_arr < w) & (y_arr >= 0) & (y_arr < h)
    x_arr, y_arr = x_arr[in_bounds], y_arr[in_bounds]

    ax.scatter(x_arr, y_arr, s=90, marker="+", color="cyan", linewidths=1.2,
               label=f"APASS reference ({len(x_arr)})", zorder=4)
    return len(x_arr)

def _render_plate_view(fits_path, mode):
    _teardown_shared_plot()

    meta = plate_db.get(str(fits_path))
    if meta is None:
        with shared_plot_output:
            clear_output(wait=True)
            print("No data available for this plate.")
        with shared_info_output:
            clear_output(wait=True)
        return

    data = astrofits.getdata(fits_path)
    hdr = astrofits.getheader(fits_path)
    wcs = WCS(hdr)

    if mode == "readonly":
        if meta.get("sources") is None:
            with shared_plot_output:
                clear_output(wait=True)
                print("No algorithm detections cached for this plate.")
            with shared_info_output:
                clear_output(wait=True)
            return
        xs = list(np.asarray(meta["sources"][meta["x_col"]], dtype=float))
        ys = list(np.asarray(meta["sources"][meta["y_col"]], dtype=float))
        flux = np.asarray(meta["sources"]["aperture_flux"], dtype=float)
        inst_mag = np.where(flux > 0, -2.5 * np.log10(np.where(flux > 0, flux, np.nan)), np.nan)
        records = _load_algorithm_labels(fits_path)
        names = [r["name"] for r in records] if records is not None and len(records) == len(xs) else ["Unlabeled"] * len(xs)
        entries = [{"x": xs[i], "y": ys[i], "name": names[i], "inst_mag": inst_mag[i]} for i in range(len(xs))]
    else:
        entries = paradigm_marks.get(str(fits_path), [])
        if meta.get("sources") is not None:
            cached_xs = np.array(meta["sources"][meta["x_col"]], dtype=float)
            cached_ys = np.array(meta["sources"][meta["y_col"]], dtype=float)
            existing_xs = np.array([e["x"] for e in entries]) if entries else np.array([])
            existing_ys = np.array([e["y"] for e in entries]) if entries else np.array([])
            new_x, new_y = [], []
            for nx, ny in zip(cached_xs, cached_ys):
                if len(existing_xs) > 0:
                    d = np.hypot(existing_xs - nx, existing_ys - ny)
                    if d.min() <= PARADIGM_CLICK_PX:
                        continue
                new_x.append(nx); new_y.append(ny)
                existing_xs = np.append(existing_xs, nx)
                existing_ys = np.append(existing_ys, ny)
            if new_x:
                ra_new, dec_new = wcs.pixel_to_world_values(np.array(new_x), np.array(new_y))
                ra_new, dec_new = np.array(ra_new, dtype=float), np.array(dec_new, dtype=float)
                for x, y, ra_i, dec_i in zip(new_x, new_y, ra_new, dec_new):
                    entries.append({"x": float(x), "y": float(y), "ra": float(ra_i), "dec": float(dec_i),
                                     "label": "", "source": "scan"})
                paradigm_marks[str(fits_path)] = entries
                save_paradigm_marks()

    with shared_plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 9))
        ax.imshow(data, origin="lower", cmap="gray")
        edit_hint = "  [left-click: label/add, right-click: remove]" if mode == "editable" else "  [click a star for details]"
        ax.set_title(f"{fits_path.name}{edit_hint}", fontsize=10)

        highlight_marker, = ax.plot([], [], "o", markersize=22, markerfacecolor="none",
                                     markeredgecolor="#FFD700", markeredgewidth=2.5, zorder=5)
        annotation = ax.annotate("", xy=(0, 0), xytext=(15, 15), textcoords="offset points",
                                  color="black", fontsize=9,
                                  bbox=dict(boxstyle="round", fc="yellow", alpha=0.9),
                                  arrowprops=dict(arrowstyle="->"))
        annotation.set_visible(False)

        if error_toggle.value == "Show Errors" and meta.get("errors"):
            annotate_errors(ax, meta["errors"])

        n_apass_shown = 0
        if apass_overlay_toggle.value == "Show APASS Reference":
            n_apass_shown = _overlay_apass_reference(ax, wcs, data.shape)

        if n_apass_shown > 0:
            ax.legend(loc="upper right", fontsize=7, framealpha=0.6, facecolor="black", labelcolor="white", edgecolor="gray")

        _viewer_state.update({
            "fig": fig, "ax": ax, "fits_path": fits_path, "mode": mode,
            "entries": entries, "wcs": wcs, "highlight_marker": highlight_marker,
            "annotation": annotation, "scatter": None, "cids": [],
        })
        _redraw_scatter()

        cids = []
        if mode == "readonly":
            cids.append(fig.canvas.mpl_connect("pick_event", _on_pick))
        else:
            cids.append(fig.canvas.mpl_connect("button_press_event", _on_paradigm_click))
        _viewer_state["cids"] = cids
        plt.show()

    with shared_info_output:
        clear_output(wait=True)
        apass_note = f"  |  {n_apass_shown} APASS reference star(s) shown" if apass_overlay_toggle.value == "Show APASS Reference" else ""
        if mode == "editable":
            print(f"Click a red/orange marker to label it, empty space to add one, or right-click a marker to remove it.{apass_note}")
        elif not entries:
            print(f"No stars to show for this plate.{apass_note}")
        else:
            print(f"{len(entries)} star(s) shown. Click one for details.{apass_note}")

def _on_review_select(change):
    fp = review_plate_dropdown.value
    if fp is None: return
    _render_plate_view(fp, "readonly")

def _on_paradigm_select(change):
    fp = paradigm_plate_dropdown.value
    if fp is None: return
    _render_plate_view(fp, "editable")
    paradigm_status.value = f"Editing {fp.name}"

def _on_toggle_change(change):
    fp = _viewer_state.get("fits_path")
    mode = _viewer_state.get("mode")
    if fp is not None and mode is not None:
        _render_plate_view(fp, mode)

review_plate_dropdown.observe(_on_review_select, names="value")
paradigm_plate_dropdown.observe(_on_paradigm_select, names="value")
error_toggle.observe(_on_toggle_change, names="value")
apass_overlay_toggle.observe(_on_toggle_change, names="value")

# Panel layout 

review_panel = widgets.VBox([
    widgets.HTML("<b>Review uncertain plates</b>"),
    review_instructions,
    review_plate_dropdown, review_status,
    widgets.HBox([approve_btn, reject_btn]),
])

clear_btn = widgets.Button(description="Clear Plate", button_style="danger")
train_btn = widgets.Button(description="Build Training Set & Train Model", button_style="success")
train_status = widgets.Label(value="")
train_output = widgets.Output()
paradigm_status = widgets.Label(value="")
paradigm_instructions = widgets.HTML(
    "<small>Stars are already shown from the scan -- no separate auto-load step "
    "needed. Left-click a red/orange marker to label it (a name box appears -- type "
    "a name or wait for the auto-suggestion, then Confirm or Skip), left-click empty "
    "space to add a manual detection, right-click a marker to remove it. Cyan '+' "
    "markers are APASS-catalogued stars shown as a reference only -- they don't "
    "affect training. 'Clear Plate' removes every saved label for this plate. Label "
    "as many objects as you like, then click 'Build Training Set & Train Model' "
    "once.</small>"
)

def _clear_plate(_):
    fp = _viewer_state.get("fits_path")
    if fp is None or _viewer_state.get("mode") != "editable":
        with shared_info_output:
            clear_output(wait=True); print("No paradigm plate is currently open.")
        return
    n_markers = len(_viewer_state["entries"])
    paradigm_marks.pop(str(fp), None)
    save_paradigm_marks()
    # Reset the display directly instead of calling _render_plate_view,
    # which would immediately re-populate markers from the cached scan
    # detections again (the same "no auto-load needed" convenience that
    # fills a paradigm plate the first time it's opened) -- that's what
    # made Clear Plate look like it did nothing.
    _viewer_state["entries"] = []
    _redraw_scatter()
    with shared_info_output:
        clear_output(wait=True); print(f"Cleared {n_markers} marker(s) from this plate.")

def _on_train_click(_):
    n_marked = sum(1 for v in paradigm_marks.values() if any(e.get("label") for e in v))
    if n_marked == 0:
        train_status.value = "Label at least one star on a paradigm plate before training."
        return
    train_status.value = "Building training set and training model... (see output below)"
    with train_output:
        clear_output(wait=True)
        build_dataset_and_train()  # defined in Cell 2
    train_status.value = "Done -- see training log and loss curve below."

clear_btn.on_click(_clear_plate)
train_btn.on_click(_on_train_click)

paradigm_panel = widgets.VBox([
    widgets.HTML("<b>Paradigm plate labeling</b>"),
    category_filter, search_box, paradigm_plate_dropdown, paradigm_dropdown_status,
    clear_btn,
    paradigm_instructions, paradigm_status,
    train_btn, train_status, train_output,
])

refresh_filter_options()
rebuild_paradigm_dropdown()

display(review_panel)
display(paradigm_panel)
display(widgets.HBox([error_toggle, apass_overlay_toggle]))
display(widgets.HTML("<b>Plate Viewer</b> (shared by Review / Paradigm labeling above)"))
display(shared_plot_output)
display(shared_info_output)

_refresh_review_queue()

HTML(value='<b>Plate Viewer</b> (shared by Review / Paradigm labeling above)')

Output()

Output()

In [ ]:
# After running, the following is generated : 
#   - .png : the plates as normalized 8-bit grayscale images.
#   - _mask.png : a black image with white boxes at every star position (ground truth for the UNet's segmentation training).
#   - .json : a list of dicts, one per star on that plate, each shaped like {"x": 412.3, "y": 187.9, "ra": 237.1442, "dec": 28.1571, "name": "R CrB"}
#   * .pth : the trained PyTorch UNet weights (star/not-star segmentation only, this file has no knowledge of star names, only pixel positions).
#   [Not file, but can be saved] A plot of the losses over time/epochs, to see if the runs are improving the CNN to the model/paradigm plate.

In [ ]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 4

import torch
from scipy import ndimage as ndi
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

PRED_MASK_THRESHOLD = 0.5  # model output at or above this counts as a star
MIN_BLOB_AREA = 4  # ignore predicted specks smaller than this many pixels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_trained_model(model_path=None):
    if model_path is None:
        model_path = output_dir / "pytorch_trial_1.pth"
    model = UNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"Loaded model from {model_path}")
    return model

_inference_model = {"model": None}

def _get_model():
    if _inference_model["model"] is None:
        _inference_model["model"] = load_trained_model()
    return _inference_model["model"]

def _predict_mask(fits_path, model):
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)

    img_t = torch.tensor(img_8bit.astype(np.float32) / 255.0).unsqueeze(0)
    orig_h, orig_w = img_t.shape[1], img_t.shape[2]
    img_t = pad_to_multiple(img_t).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(img_t)
    pred = pred.squeeze().cpu().numpy()
    pred = pred[:orig_h, :orig_w]
    return img_8bit, pred

def _mask_to_points(pred_mask, threshold=PRED_MASK_THRESHOLD, min_area=MIN_BLOB_AREA):
    binary = (pred_mask >= threshold).astype(np.uint8)
    labeled, n_obj = ndi.label(binary)
    xs, ys = [], []
    for obj_id in range(1, n_obj + 1):
        region = labeled == obj_id
        area = region.sum()
        if area < min_area:
            continue
        coords = np.argwhere(region)
        cy, cx = coords.mean(axis=0)
        xs.append(float(cx)); ys.append(float(cy))
    return xs, ys

def run_inference_on_plate(fits_path, identify=True, save=True):
    model = _get_model()
    img_8bit, pred_mask = _predict_mask(fits_path, model)
    xs, ys = _mask_to_points(pred_mask)

    records = None
    if identify and len(xs) > 0:
        records = _identify_sources(fits_path, xs, ys)
        if save:
            out_path = labels_dir / (fits_path.stem + "_predicted_labels.json")
            with open(out_path, "w") as f:
                json.dump(records, f, indent=2)

    return xs, ys, records

# Load the trained model immediately when this cell runs, not lazily on
# first click.
_model_ready = False
try:
    _get_model()
    _model_ready = True
except FileNotFoundError:
    print(f"No trained model found yet at {output_dir / 'pytorch_trial_1.pth'}.")
    print("Train one first via Cell 3's 'Build Training Set & Train Model' button, then re-run this cell.")
except Exception as e:
    print(f"[MODEL LOAD ERROR] {e}")

# ---------- Full analysis panel: algorithm detections + errors,
# model predictions, linearity check, FWHM analysis -- one shared
# filter/search/dropdown and one shared interactive plot area, same
# architecture as Cell 3's viewer. ----------

analyze_mode = widgets.ToggleButtons(
    options=["Detections & Errors", "Model Predictions", "Linearity Check", "FWHM Analysis"],
    value="Detections & Errors", description="Mode:"
)
analyze_error_toggle = widgets.ToggleButtons(options=["Show Errors", "Hide Errors"], value="Show Errors", description="Errors:")
analyze_category_filter = widgets.Dropdown(description="Filter:")
analyze_search_box = widgets.Text(description="Search:", placeholder="filter by filename")
analyze_dropdown = widgets.Dropdown(description="Plate:")
analyze_dropdown_status = widgets.Label(value="")
analyze_status = widgets.Label(value="" if _model_ready else "Model not loaded -- see message above (only affects Model Predictions mode).")
run_analysis_btn = widgets.Button(description="Run / Refresh", button_style="info")

analyze_plot_output = widgets.Output()
analyze_info_output = widgets.Output()

def rebuild_analyze_dropdown(*args):
    matches = filter_plates(analyze_category_filter.value, analyze_search_box.value)
    analyze_dropdown.options = matches
    analyze_dropdown_status.value = f"{len(matches)} matches"

analyze_category_filter.options = category_filter.options
analyze_category_filter.value = "all"
analyze_category_filter.observe(rebuild_analyze_dropdown, names="value")
analyze_search_box.observe(rebuild_analyze_dropdown, names="value")
rebuild_analyze_dropdown()

_analysis_state = {"fig": None, "ax": None, "cids": [], "scatter": None,
                    "highlight": None, "annotation": None, "entries": []}

def _teardown_analysis_plot():
    fig = _analysis_state.get("fig")
    if fig is not None:
        for cid in _analysis_state.get("cids", []):
            try:
                fig.canvas.mpl_disconnect(cid)
            except Exception:
                pass
        plt.close(fig)
    _analysis_state.update({"fig": None, "ax": None, "cids": [], "highlight": None,
                             "annotation": None, "scatter": None})

def _on_analysis_pick(event):
    if event.artist != _analysis_state["scatter"]:
        return
    i = event.ind[0]
    e = _analysis_state["entries"][i]
    hl = _analysis_state["highlight"]
    hl.set_data([e["x"]], [e["y"]])
    hl.set_visible(True)
    ann = _analysis_state["annotation"]
    ann.xy = (e["x"], e["y"])
    ann.set_text(e["label_text"])
    ann.set_visible(True)
    _analysis_state["fig"].canvas.draw_idle()
    with analyze_info_output:
        clear_output(wait=True)
        print(e["label_text"].replace("\n", "   |   "))

def _load_algorithm_labels(fits_path):
    label_path = labels_dir / (fits_path.stem + "_labels.json")
    if not label_path.exists():
        return None
    with open(label_path, "r") as f:
        return json.load(f)

def _load_predicted_labels(fits_path):
    label_path = labels_dir / (fits_path.stem + "_predicted_labels.json")
    if not label_path.exists():
        return None
    with open(label_path, "r") as f:
        return json.load(f)

def _new_single_panel(fits_path, data, title):
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(data, origin="lower", cmap="gray")
    ax.set_title(title, fontsize=10)
    highlight, = ax.plot([], [], "o", markersize=22, markerfacecolor="none",
                          markeredgecolor="#FFD700", markeredgewidth=2.5, zorder=5)
    annotation = ax.annotate("", xy=(0, 0), xytext=(15, 15), textcoords="offset points",
                              color="black", fontsize=9,
                              bbox=dict(boxstyle="round", fc="yellow", alpha=0.9),
                              arrowprops=dict(arrowstyle="->"))
    annotation.set_visible(False)
    return fig, ax, highlight, annotation

def _finish_panel(fig, ax, scatter, entries, highlight, annotation):
    _analysis_state.update({"fig": fig, "ax": ax, "scatter": scatter, "entries": entries,
                             "highlight": highlight, "annotation": annotation})
    cid = fig.canvas.mpl_connect("pick_event", _on_analysis_pick)
    _analysis_state["cids"] = [cid]
    plt.show()

def _render_detections_errors(fits_path, meta, data):
    if meta.get("sources") is None:
        with analyze_plot_output:
            clear_output(wait=True)
            print("No algorithm detections cached for this plate.")
        return
    xs = list(np.asarray(meta["sources"][meta["x_col"]], dtype=float))
    ys = list(np.asarray(meta["sources"][meta["y_col"]], dtype=float))
    flux = np.asarray(meta["sources"]["aperture_flux"], dtype=float)
    inst_mag = np.where(flux > 0, -2.5 * np.log10(np.where(flux > 0, flux, np.nan)), np.nan)
    records = _load_algorithm_labels(fits_path)
    names = [r["name"] for r in records] if records is not None and len(records) == len(xs) else ["Unlabeled"] * len(xs)

    entries = []
    for i in range(len(xs)):
        lines = [names[i]]
        if np.isfinite(inst_mag[i]):
            lines.append(f"inst={inst_mag[i]:.2f}")
        entries.append({"x": xs[i], "y": ys[i], "label_text": "\n".join(lines)})

    with analyze_plot_output:
        clear_output(wait=True)
        fig, ax, highlight, annotation = _new_single_panel(
            fits_path, data, f"{fits_path.name}  [{meta.get('quality','?')}]  (click a star for details)"
        )
        scatter = ax.scatter(xs, ys, s=60, facecolors="none", edgecolors="red", picker=True, pickradius=8)
        if analyze_error_toggle.value == "Show Errors" and meta.get("errors"):
            annotate_errors(ax, meta["errors"])
        _finish_panel(fig, ax, scatter, entries, highlight, annotation)

    with analyze_info_output:
        clear_output(wait=True)
        errs = meta.get("errors", {})
        print(f"quality={meta.get('quality')}  n_sources={len(xs)}  "
              f"scratches={len(errs.get('scratches', []))} trailing={len(errs.get('trailing', []))} "
              f"saturation={len(errs.get('saturation', []))} dust={len(errs.get('dust', []))} "
              f"dead_zone={errs.get('dead_zone_fraction', 0)*100:.1f}% "
              f"sat_area={errs.get('saturation_area_fraction', 0)*100:.1f}%")
        print(f"{len(xs)} star(s) shown. Click one for details.")

def _render_model_predictions(fits_path, meta, data):
    records = _load_predicted_labels(fits_path)
    if records is None:
        with analyze_plot_output:
            clear_output(wait=True)
            print("No model predictions saved for this plate yet. Click 'Run / Refresh' below.")
        with analyze_info_output:
            clear_output(wait=True)
        return

    entries = [{"x": r["x"], "y": r["y"], "label_text": r["name"]} for r in records]
    xs = [e["x"] for e in entries]
    ys = [e["y"] for e in entries]

    with analyze_plot_output:
        clear_output(wait=True)
        fig, ax, highlight, annotation = _new_single_panel(
            fits_path, data, f"{fits_path.name}  [model predictions]  (click a star for details)"
        )
        scatter = ax.scatter(xs, ys, s=60, facecolors="none", edgecolors="red", picker=True, pickradius=8)
        if analyze_error_toggle.value == "Show Errors" and meta.get("errors"):
            annotate_errors(ax, meta["errors"])
        _finish_panel(fig, ax, scatter, entries, highlight, annotation)

    with analyze_info_output:
        clear_output(wait=True)
        n_named = sum(1 for e in entries if e["label_text"] not in ("Unknown", ""))
        print(f"{len(entries)} predicted star(s), {n_named} named. Click one for details.")

def _render_linearity(fits_path, meta, data):
    if meta.get("sources") is None:
        with analyze_plot_output:
            clear_output(wait=True)
            print("No algorithm detections cached for this plate. Run Cell 1's prescan first.")
        return

    hdr = astrofits.getheader(fits_path)
    wcs = WCS(hdr)
    sources = meta["sources"]
    xs = np.array(sources[meta["x_col"]], dtype=float)
    ys = np.array(sources[meta["y_col"]], dtype=float)
    flux = np.array(sources["aperture_flux"], dtype=float)
    ra, dec = wcs.pixel_to_world_values(xs, ys)
    ra, dec = np.array(ra, dtype=float), np.array(dec, dtype=float)

    apass_catalog = get_plate_catalog_apass(wcs, data.shape, apass_cache)
    save_apass_cache()
    catalog_mag = match_detected_sources_apass(ra, dec, apass_catalog)
    inst_mag = np.where(flux > 0, -2.5 * np.log10(np.where(flux > 0, flux, np.nan)), np.nan)
    fit = compute_linearity_fit(inst_mag, catalog_mag)

    entries = []
    for i in range(len(xs)):
        lines = ["source"]
        if np.isfinite(inst_mag[i]):
            lines.append(f"inst={inst_mag[i]:.2f}")
        if np.isfinite(catalog_mag[i]):
            lines.append(f"APASS B={catalog_mag[i]:.2f}")
        else:
            lines.append("no APASS match")
        entries.append({"x": xs[i], "y": ys[i], "label_text": "\n".join(lines)})

    with analyze_plot_output:
        clear_output(wait=True)
        fig, (ax_img, ax_fit) = plt.subplots(1, 2, figsize=(16, 7))
        ax_img.imshow(data, origin="lower", cmap="gray")
        ax_img.set_title(f"{fits_path.name}  (click a star for details)")
        scatter = ax_img.scatter(xs, ys, s=40, facecolors="none", edgecolors="red", picker=True, pickradius=8)
        highlight, = ax_img.plot([], [], "o", markersize=20, markerfacecolor="none",
                                  markeredgecolor="#FFD700", markeredgewidth=2.5, zorder=5)
        annotation = ax_img.annotate("", xy=(0, 0), xytext=(15, 15), textcoords="offset points",
                                      color="black", fontsize=9,
                                      bbox=dict(boxstyle="round", fc="yellow", alpha=0.9),
                                      arrowprops=dict(arrowstyle="->"))
        annotation.set_visible(False)
        if analyze_error_toggle.value == "Show Errors" and meta.get("errors"):
            annotate_errors(ax_img, meta["errors"])

        matched = np.isfinite(catalog_mag) & np.isfinite(inst_mag)
        ax_fit.scatter(catalog_mag[matched], inst_mag[matched], s=25, color="steelblue", alpha=0.7, label="Matched sources")
        if fit is not None:
            x_fit = np.linspace(np.nanmin(catalog_mag[matched]), np.nanmax(catalog_mag[matched]), 100)
            y_fit = fit["slope"] * x_fit + fit["intercept"]
            ax_fit.plot(x_fit, y_fit, color="red", linewidth=1.5,
                        label=f"Fit: slope={fit['slope']:.3f}, n={fit['n_used']}, RMS={fit['rms']:.3f} mag")
            ax_fit.set_title(f"Linearity Check (RMS = {fit['rms']:.3f} mag)")
        else:
            ax_fit.set_title("Linearity Check (not enough APASS matches)")
        ax_fit.set_xlabel("APASS B magnitude")
        ax_fit.set_ylabel("Instrumental magnitude")
        ax_fit.invert_yaxis()
        ax_fit.legend(fontsize=8)
        ax_fit.grid(alpha=0.3)

        _finish_panel(fig, ax_img, scatter, entries, highlight, annotation)

    with analyze_info_output:
        clear_output(wait=True)
        if fit is not None:
            print(f"{np.sum(matched)} of {len(xs)} sources matched APASS. Fit RMS = {fit['rms']:.3f} mag. Click a star for details.")
        else:
            print(f"{np.sum(matched)} of {len(xs)} sources matched APASS -- too few for a fit.")

def _render_fwhm(fits_path, meta, data):
    if meta.get("sources") is None:
        with analyze_plot_output:
            clear_output(wait=True)
            print("No algorithm detections cached for this plate. Run Cell 1's prescan first.")
        return

    sources = meta["sources"]
    xs = np.array(sources[meta["x_col"]], dtype=float)
    ys = np.array(sources[meta["y_col"]], dtype=float)
    fwhm_vals = compute_fwhm_for_sources(data, xs, ys)

    entries = []
    for i in range(len(xs)):
        lines = ["source"]
        if np.isfinite(fwhm_vals[i]):
            lines.append(f"FWHM={fwhm_vals[i]:.2f}px")
        else:
            lines.append("FWHM fit failed")
        entries.append({"x": xs[i], "y": ys[i], "label_text": "\n".join(lines)})

    with analyze_plot_output:
        clear_output(wait=True)
        fig, (ax_img, ax_hist) = plt.subplots(1, 2, figsize=(16, 7))
        ax_img.imshow(data, origin="lower", cmap="gray")
        ax_img.set_title(f"{fits_path.name}  (click a star for details)")
        scatter = ax_img.scatter(xs, ys, s=40, facecolors="none", edgecolors="red", picker=True, pickradius=8)
        highlight, = ax_img.plot([], [], "o", markersize=20, markerfacecolor="none",
                                  markeredgecolor="#FFD700", markeredgewidth=2.5, zorder=5)
        annotation = ax_img.annotate("", xy=(0, 0), xytext=(15, 15), textcoords="offset points",
                                      color="black", fontsize=9,
                                      bbox=dict(boxstyle="round", fc="yellow", alpha=0.9),
                                      arrowprops=dict(arrowstyle="->"))
        annotation.set_visible(False)
        if analyze_error_toggle.value == "Show Errors" and meta.get("errors"):
            annotate_errors(ax_img, meta["errors"])

        valid = fwhm_vals[np.isfinite(fwhm_vals)]
        if len(valid) > 0:
            ax_hist.hist(valid, bins=30, color="steelblue", alpha=0.8)
            med = float(np.median(valid))
            ax_hist.axvline(med, color="red", linewidth=1.5, label=f"Median = {med:.2f}px")
            ax_hist.set_title(f"FWHM distribution (n={len(valid)} of {len(xs)} fit successfully)")
            ax_hist.legend(fontsize=8)
        else:
            ax_hist.set_title("FWHM distribution (no successful fits)")
        ax_hist.set_xlabel("FWHM (pixels)")
        ax_hist.set_ylabel("Count")
        ax_hist.grid(alpha=0.3)

        _finish_panel(fig, ax_img, scatter, entries, highlight, annotation)

    with analyze_info_output:
        clear_output(wait=True)
        if len(valid) > 0:
            print(f"Median FWHM = {np.median(valid):.2f}px, std = {np.std(valid):.2f}px, "
                  f"range [{valid.min():.2f}, {valid.max():.2f}]px. Click a star for details.")
        else:
            print("No FWHM fits succeeded on this plate.")

def show_analysis(change=None):
    fp = analyze_dropdown.value
    if fp is None:
        with analyze_plot_output:
            clear_output(wait=True)
            print("No plate selected.")
        return
    meta = plate_db.get(str(fp))
    if meta is None:
        with analyze_plot_output:
            clear_output(wait=True)
            print("This plate hasn't been scanned yet. Run Cell 1's prescan first.")
        return

    _teardown_analysis_plot()
    analyze_status.value = "Rendering..."
    data = astrofits.getdata(fp)

    if analyze_mode.value == "Detections & Errors":
        _render_detections_errors(fp, meta, data)
    elif analyze_mode.value == "Model Predictions":
        _render_model_predictions(fp, meta, data)
    elif analyze_mode.value == "Linearity Check":
        _render_linearity(fp, meta, data)
    else:
        _render_fwhm(fp, meta, data)

    analyze_status.value = "Done."

def _on_run_analysis_click(_):
    fp = analyze_dropdown.value
    if fp is None:
        analyze_status.value = "No plate selected."
        return
    if analyze_mode.value == "Model Predictions":
        if not _model_ready:
            analyze_status.value = "No trained model loaded -- train one in Cell 3, then re-run this cell."
            return
        analyze_status.value = "Running model..."
        xs, ys, records = run_inference_on_plate(fp)
        n_named = sum(1 for r in records if r["name"] != "Unknown") if records else 0
        analyze_status.value = f"Done: {len(xs)} star(s) found, {n_named} named."
        show_analysis()
    elif analyze_mode.value == "Linearity Check":
        analyze_status.value = "Querying APASS and computing fit..."
        show_analysis()
    elif analyze_mode.value == "FWHM Analysis":
        analyze_status.value = "Computing FWHM fits..."
        show_analysis()
    else:
        show_analysis()

analyze_dropdown.observe(show_analysis, names="value")
analyze_mode.observe(show_analysis, names="value")
analyze_error_toggle.observe(show_analysis, names="value")
run_analysis_btn.on_click(_on_run_analysis_click)

display(widgets.VBox([
    widgets.HTML("<b>Full plate analysis</b> -- detections & errors, model predictions, linearity check, FWHM."),
    analyze_mode, analyze_error_toggle,
    analyze_category_filter, analyze_search_box, analyze_dropdown, analyze_dropdown_status,
    run_analysis_btn, analyze_status,
    analyze_plot_output, analyze_info_output,
]))

if analyze_dropdown.options:
    analyze_dropdown.value = analyze_dropdown.options[0][1]